# Patient-condition sensitivity analysis

**Purpose:** quantify repeated sampling within patients and compare the published sample-level condition analyses with patient-condition-level sensitivity analyses.

This notebook is diagnostic. It does not overwrite any canonical result. All outputs are written to `metadata/sensitivity/patient_condition_sensitivity/`.

Scope:
- final CNV-informed compartment proportions;
- malignant-state proportions;
- malignant-cell program scores;
- overall-malignant pseudobulk condition contrasts;
- explicit accounting of repeated samples and patients spanning multiple conditions.

The patient-condition unit is defined by dataset, patient, and disease condition. Counts are summed across samples belonging to the same patient-condition unit. Program scores are combined using malignant-cell counts as weights. Because LGG and rGBM contain few independent patients, all patient-level results are interpreted as sensitivity analyses rather than definitive inferential evidence.

In [1]:
from pathlib import Path
from itertools import combinations
import json
import platform
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "metadata").exists():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "metadata").exists():
    raise FileNotFoundError("Run this notebook from within the repository.")

METADATA_DIR = PROJECT_DIR / "metadata"
OUT_DIR = METADATA_DIR / "sensitivity" / "patient_condition_sensitivity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITION_ORDER = ["GBM", "LGG", "ndGBM", "rGBM"]
ALPHA = 0.05

PATHS = {
    "compartment": METADATA_DIR / "DA_sample_level_compartment_proportions_HVG8000_r0_6.tsv",
    "compartment_global": METADATA_DIR / "DA_global_kruskal_compartment_sample_level_HVG8000_r0_6.tsv",
    "compartment_pairwise": METADATA_DIR / "DA_pairwise_mannwhitney_compartment_sample_level_HVG8000_r0_6.tsv",
    "state": METADATA_DIR / "malignant_state_sample_level_proportions_HVG8000_r0_6.tsv",
    "state_global": METADATA_DIR / "malignant_state_global_kruskal_sample_level_HVG8000_r0_6.tsv",
    "state_pairwise": METADATA_DIR / "malignant_state_pairwise_mannwhitney_sample_level_HVG8000_r0_6.tsv",
    "program": METADATA_DIR / "condition_malignant_pseudobulk_DE_HVG8000_r0_6_sample_level_program_scores.tsv",
    "program_global": METADATA_DIR / "condition_malignant_pseudobulk_DE_HVG8000_r0_6_program_score_global_kruskal.tsv",
    "program_pairwise": METADATA_DIR / "condition_malignant_pseudobulk_DE_HVG8000_r0_6_program_score_pairwise_mannwhitney.tsv",
    "pb_counts": METADATA_DIR / "condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_counts_pseudobulk.tsv.gz",
    "pb_qc": METADATA_DIR / "condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_sample_QC.tsv",
    "pb_sample_summary": METADATA_DIR / "condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_test_summary.tsv",
}
missing=[str(p) for p in PATHS.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

def write_tsv(df, name):
    path=OUT_DIR/name
    df.to_csv(path, sep="	", index=False)
    return path

def bh(values):
    x=np.asarray(values, dtype=float)
    out=np.full(x.shape, np.nan)
    ok=np.isfinite(x)
    if ok.any():
        out[ok]=multipletests(x[ok], method="fdr_bh")[1]
    return out

def key_frame(df, dataset, patient, condition, sample):
    out=df[[dataset, patient, condition, sample]].drop_duplicates().copy()
    out.columns=["dataset","patient","condition","sample_id"]
    for c in out.columns:
        out[c]=out[c].astype(str)
    out["patient_condition_id"]=out["dataset"]+"||"+out["patient"]+"||"+out["condition"]
    return out

print("Project:", PROJECT_DIR)
print("Output:", OUT_DIR)


Project: G:\Repository\glioma-cnv-single-cell-atlas
Output: G:\Repository\glioma-cnv-single-cell-atlas\metadata\sensitivity\patient_condition_sensitivity


## 1. Repeated-sampling audit

In [2]:
comp=pd.read_csv(PATHS["compartment"], sep="	")
sample_map=key_frame(comp, "dataset", "patient", "condition", "unit_id")
if sample_map["sample_id"].duplicated().any():
    raise AssertionError("A sample maps to more than one dataset/patient/condition combination.")

patient_audit=(sample_map.groupby(["dataset","patient"], observed=True)
    .agg(n_samples=("sample_id","nunique"), n_conditions=("condition","nunique"),
         conditions=("condition",lambda s:";".join(sorted(set(s)))))
    .reset_index())
condition_units=(sample_map.groupby("condition", observed=True)
    .agg(n_samples=("sample_id","nunique"), n_patients=("patient_condition_id","nunique"))
    .reindex(CONDITION_ORDER).reset_index())

audit_summary=pd.DataFrame([
    {"metric":"samples", "value":sample_map.sample_id.nunique()},
    {"metric":"dataset-scoped patients", "value":sample_map[["dataset","patient"]].drop_duplicates().shape[0]},
    {"metric":"patients with multiple samples", "value":int((patient_audit.n_samples>1).sum())},
    {"metric":"patients spanning multiple conditions", "value":int((patient_audit.n_conditions>1).sum())},
])
write_tsv(patient_audit, "patient_repeated_sampling_audit.tsv")
write_tsv(condition_units, "condition_independent_unit_counts.tsv")
write_tsv(audit_summary, "audit_summary.tsv")
display(audit_summary)
display(condition_units)
display(patient_audit.loc[patient_audit.n_samples>1])


,metric,value
0,samples,83
1,dataset-scoped patients,50
2,patients with multiple samples,6
3,patients spanning multiple conditions,5


,condition,n_samples,n_patients
0,GBM,39,39
1,LGG,4,2
2,ndGBM,22,11
3,rGBM,18,5


,dataset,patient,n_samples,n_conditions,conditions
17,DS3_GSE182109,01,8,2,ndGBM;rGBM
18,DS3_GSE182109,02,8,2,ndGBM;rGBM
19,DS3_GSE182109,03,7,3,LGG;ndGBM;rGBM
20,DS3_GSE182109,04,8,3,LGG;ndGBM;rGBM
21,DS3_GSE182109,05,4,2,ndGBM;rGBM
27,DS3_GSE182109,11,4,1,ndGBM


## 2. Patient-condition compartment analysis

In [3]:
comp=comp.merge(sample_map[["sample_id","patient_condition_id"]], left_on="unit_id", right_on="sample_id", how="left", validate="many_to_one")
patient_comp=(comp.groupby(["patient_condition_id","dataset","patient","condition","feature","feature_type"], observed=True)
    .agg(n_cells_feature=("n_cells_feature","sum"), n_cells_total=("n_cells_total","sum"), n_samples=("unit_id","nunique"))
    .reset_index())
patient_comp["proportion"]=patient_comp.n_cells_feature/patient_comp.n_cells_total
write_tsv(patient_comp, "patient_condition_compartment_proportions.tsv")

global_rows=[]
for feature, sub in patient_comp.groupby("feature", observed=True):
    groups=[sub.loc[sub.condition.eq(c),"proportion"].dropna().to_numpy() for c in CONDITION_ORDER]
    if all(len(x)>=2 for x in groups):
        stat,p=stats.kruskal(*groups)
    else:
        stat,p=np.nan,np.nan
    global_rows.append({"feature":feature,"statistic":stat,"p_value":p,
                        **{f"n_{c}":len(x) for c,x in zip(CONDITION_ORDER,groups)}})
comp_global=pd.DataFrame(global_rows)
comp_global["q_value_bh"]=bh(comp_global.p_value)
write_tsv(comp_global,"patient_condition_compartment_global_kruskal.tsv")

pair_rows=[]
for feature, sub in patient_comp.groupby("feature", observed=True):
    for a,b in combinations(CONDITION_ORDER,2):
        x=sub.loc[sub.condition.eq(a),"proportion"].dropna().to_numpy()
        y=sub.loc[sub.condition.eq(b),"proportion"].dropna().to_numpy()
        p=stats.mannwhitneyu(x,y,alternative="two-sided").pvalue if len(x)>=2 and len(y)>=2 else np.nan
        pair_rows.append({"feature":feature,"condition_1":a,"condition_2":b,"n_1":len(x),"n_2":len(y),
                          "median_1":np.median(x) if len(x) else np.nan,"median_2":np.median(y) if len(y) else np.nan,"p_value":p})
comp_pair=pd.DataFrame(pair_rows)
comp_pair["q_value_bh"]=comp_pair.groupby(["condition_1","condition_2"])["p_value"].transform(bh)
write_tsv(comp_pair,"patient_condition_compartment_pairwise_mannwhitney.tsv")
display(comp_global)


,feature,statistic,p_value,n_GBM,n_LGG,n_ndGBM,n_rGBM,q_value_bh
0,cycling_malignant_like_candidate,2.081598,0.555644,39,2,9,5,0.689716
1,malignant_like,8.391073,0.038584,39,2,11,5,0.077169
2,non_malignant_or_diploid_leaning,1.467823,0.689716,39,2,10,5,0.689716
3,review,11.175690,0.010813,39,2,11,5,0.043251


## 3. Patient-condition malignant-state analysis

In [4]:
state=pd.read_csv(PATHS["state"], sep="	")
state=state.merge(sample_map[["sample_id","patient_condition_id"]], on="sample_id", how="left", validate="many_to_one")
patient_state=(state.groupby(["patient_condition_id","core_dataset","patient_id","condition","malignant_state"], observed=True)
    .agg(n_state_cells=("n_state_cells","sum"), n_pool=("n_malignant_state_analysis_cells","sum"),
         n_all=("n_total_cells_in_sample","sum"), n_samples=("sample_id","nunique"))
    .reset_index())
patient_state["fraction_within_malignant_state_pool"]=patient_state.n_state_cells/patient_state.n_pool
patient_state["fraction_of_all_cells"]=patient_state.n_state_cells/patient_state.n_all
write_tsv(patient_state,"patient_condition_malignant_state_proportions.tsv")

grows=[]; prows=[]
for feature, sub in patient_state.groupby("malignant_state", observed=True):
    groups=[sub.loc[sub.condition.eq(c),"fraction_within_malignant_state_pool"].dropna().to_numpy() for c in CONDITION_ORDER]
    stat,p=stats.kruskal(*groups) if all(len(x)>=2 for x in groups) else (np.nan,np.nan)
    grows.append({"malignant_state":feature,"statistic":stat,"p_value":p,**{f"n_{c}":len(x) for c,x in zip(CONDITION_ORDER,groups)}})
    for a,b in combinations(CONDITION_ORDER,2):
        x=sub.loc[sub.condition.eq(a),"fraction_within_malignant_state_pool"].dropna().to_numpy()
        y=sub.loc[sub.condition.eq(b),"fraction_within_malignant_state_pool"].dropna().to_numpy()
        p2=stats.mannwhitneyu(x,y,alternative="two-sided").pvalue if len(x)>=2 and len(y)>=2 else np.nan
        prows.append({"malignant_state":feature,"condition_1":a,"condition_2":b,"n_1":len(x),"n_2":len(y),
                      "median_1":np.median(x) if len(x) else np.nan,"median_2":np.median(y) if len(y) else np.nan,"p_value":p2})
state_global=pd.DataFrame(grows); state_global["q_value_bh"]=bh(state_global.p_value)
state_pair=pd.DataFrame(prows); state_pair["q_value_bh"]=state_pair.groupby(["condition_1","condition_2"])["p_value"].transform(bh)
write_tsv(state_global,"patient_condition_state_global_kruskal.tsv")
write_tsv(state_pair,"patient_condition_state_pairwise_mannwhitney.tsv")
display(state_global)


,malignant_state,statistic,p_value,n_GBM,n_LGG,n_ndGBM,n_rGBM,q_value_bh
0,AC_like_reactive_malignant,2.768317,0.428742,39,2,11,5,0.428742
1,NPC_neuronal_like_malignant,3.131141,0.371844,39,2,11,5,0.428742
2,OPC_NPC_like_malignant,5.827789,0.120296,39,2,11,5,0.318935
3,cycling_malignant_candidate,5.692385,0.127574,39,2,11,5,0.318935
4,hypoxia_stress_malignant,4.453047,0.216517,39,2,11,5,0.360861


## 4. Patient-condition program-score analysis

In [5]:
program=pd.read_csv(PATHS["program"], sep="	")
program=program.merge(sample_map[["sample_id","dataset","patient","condition","patient_condition_id"]], on=["sample_id","condition"], how="left", validate="one_to_one")
score_cols=[c for c in program.columns if c.endswith("_score")]
rows=[]
for key,sub in program.groupby(["patient_condition_id","dataset","patient","condition"], observed=True):
    w=pd.to_numeric(sub.n_cells, errors="coerce").fillna(0).to_numpy(float)
    row={"patient_condition_id":key[0],"dataset":key[1],"patient":key[2],"condition":key[3],"n_samples":sub.sample_id.nunique(),"n_cells":w.sum()}
    for col in score_cols:
        v=pd.to_numeric(sub[col],errors="coerce").to_numpy(float)
        ok=np.isfinite(v)&(w>0)
        row[col]=np.average(v[ok],weights=w[ok]) if ok.any() else np.nan
    rows.append(row)
patient_program=pd.DataFrame(rows)
write_tsv(patient_program,"patient_condition_program_scores.tsv")

grows=[]; prows=[]
for score in score_cols:
    groups=[patient_program.loc[patient_program.condition.eq(c),score].dropna().to_numpy() for c in CONDITION_ORDER]
    stat,p=stats.kruskal(*groups) if all(len(x)>=2 for x in groups) else (np.nan,np.nan)
    grows.append({"program":score,"statistic":stat,"p_value":p,**{f"n_{c}":len(x) for c,x in zip(CONDITION_ORDER,groups)}})
    for a,b in combinations(CONDITION_ORDER,2):
        x=patient_program.loc[patient_program.condition.eq(a),score].dropna().to_numpy(); y=patient_program.loc[patient_program.condition.eq(b),score].dropna().to_numpy()
        p2=stats.mannwhitneyu(x,y,alternative="two-sided").pvalue if len(x)>=2 and len(y)>=2 else np.nan
        prows.append({"program":score,"condition_1":a,"condition_2":b,"n_1":len(x),"n_2":len(y),"p_value":p2})
prog_global=pd.DataFrame(grows); prog_global["q_value_bh"]=bh(prog_global.p_value)
prog_pair=pd.DataFrame(prows); prog_pair["q_value_bh"]=prog_pair.groupby(["condition_1","condition_2"])["p_value"].transform(bh)
write_tsv(prog_global,"patient_condition_program_global_kruskal.tsv")
write_tsv(prog_pair,"patient_condition_program_pairwise_mannwhitney.tsv")
display(prog_global)


,program,statistic,p_value,n_GBM,n_LGG,n_ndGBM,n_rGBM,q_value_bh
0,OPC_proneural_score,4.013659,0.259993,39,2,10,5,0.363990
1,NPC_neural_score,1.895209,0.594439,39,2,10,5,0.594439
2,AC_reactive_score,1.998535,0.572711,39,2,10,5,0.594439
3,MES_like_score,6.182745,0.103050,39,2,10,5,0.303083
4,hypoxia_stress_score,5.650790,0.129893,39,2,10,5,0.303083
5,cycling_score,4.253104,0.235399,39,2,10,5,0.363990
6,immune_myeloid_review_score,6.017794,0.110748,39,2,10,5,0.303083


## 5. Overall-malignant patient-condition pseudobulk sensitivity

In [6]:
counts=pd.read_csv(PATHS["pb_counts"],sep="	")
counts=counts.rename(columns={counts.columns[0]:"sample_id"})
gene_cols=[c for c in counts.columns if c!="sample_id"]
counts=counts.merge(sample_map[["sample_id","patient_condition_id","condition"]],on="sample_id",how="left",validate="one_to_one")
if counts.patient_condition_id.isna().any():
    raise AssertionError("Some pseudobulk samples did not map to patient-condition units.")
patient_counts=counts.groupby(["patient_condition_id","condition"],observed=True)[gene_cols].sum().reset_index()
lib=patient_counts[gene_cols].sum(axis=1).to_numpy(float)
if (lib<=0).any(): raise AssertionError("Zero-library patient pseudobulk detected.")
logcpm=np.log1p(patient_counts[gene_cols].div(lib,axis=0)*1e6)
patient_log=pd.concat([patient_counts[["patient_condition_id","condition"]],logcpm],axis=1)
patient_counts.to_csv(OUT_DIR/"patient_condition_overall_malignant_counts.tsv.gz",sep="	",index=False,compression="gzip")
patient_log.to_csv(OUT_DIR/"patient_condition_overall_malignant_logCPM.tsv.gz",sep="	",index=False,compression="gzip")

summary=[]; all_results=[]
for a,b in combinations(CONDITION_ORDER,2):
    xa=logcpm.loc[patient_counts.condition.eq(a)].to_numpy(float); xb=logcpm.loc[patient_counts.condition.eq(b)].to_numpy(float)
    mean_a=np.nanmean(xa,axis=0); mean_b=np.nanmean(xb,axis=0); lfc=mean_a-mean_b
    welch=stats.ttest_ind(xa,xb,axis=0,equal_var=False,nan_policy="omit").pvalue if len(xa)>=2 and len(xb)>=2 else np.full(len(gene_cols),np.nan)
    mw=np.array([stats.mannwhitneyu(xa[:,j],xb[:,j],alternative="two-sided").pvalue for j in range(len(gene_cols))]) if len(xa)>=2 and len(xb)>=2 else np.full(len(gene_cols),np.nan)
    res=pd.DataFrame({"gene":gene_cols,"condition_A":a,"condition_B":b,"mean_logCPM_A":mean_a,"mean_logCPM_B":mean_b,"log2fc_A_minus_B":lfc,"welch_p":welch,"mw_p":mw})
    res["welch_q"]=bh(res.welch_p); res["mw_q"]=bh(res.mw_p)
    res["significant"]=(res.welch_q<ALPHA)&(res.mw_q<ALPHA)&(res.log2fc_A_minus_B.abs()>=0.5)
    all_results.append(res)
    summary.append({"contrast":f"{a}_vs_{b}","n_A":len(xa),"n_B":len(xb),"n_genes_tested":len(gene_cols),"n_significant_joint":int(res.significant.sum()),"status":"tested" if len(xa)>=2 and len(xb)>=2 else "insufficient"})
pb_results=pd.concat(all_results,ignore_index=True)
pb_summary=pd.DataFrame(summary)
pb_results.to_csv(OUT_DIR/"patient_condition_overall_malignant_pairwise_DE_all_genes.tsv.gz",sep="	",index=False,compression="gzip")
write_tsv(pb_summary,"patient_condition_overall_malignant_pairwise_DE_summary.tsv")
display(pb_summary)


,contrast,n_A,n_B,n_genes_tested,n_significant_joint,status
0,GBM_vs_LGG,39,2,15081,376,tested
1,GBM_vs_ndGBM,39,10,15081,1487,tested
2,GBM_vs_rGBM,39,5,15081,528,tested
3,LGG_vs_ndGBM,2,10,15081,0,tested
4,LGG_vs_rGBM,2,5,15081,0,tested
5,ndGBM_vs_rGBM,10,5,15081,0,tested


## 6. Readiness summary and manifest

In [7]:
checks=pd.DataFrame([
    {"check":"all canonical inputs found","pass":all(p.exists() for p in PATHS.values())},
    {"check":"sample mapping unique","pass":not sample_map.sample_id.duplicated().any()},
    {"check":"four conditions present","pass":set(sample_map.condition)==set(CONDITION_ORDER)},
    {"check":"repeated patients detected","pass":bool((patient_audit.n_samples>1).any())},
    {"check":"patient-condition compartment results written","pass":not comp_global.empty},
    {"check":"patient-condition state results written","pass":not state_global.empty},
    {"check":"patient-condition program results written","pass":not prog_global.empty},
    {"check":"patient-condition pseudobulk results written","pass":not pb_summary.empty},
])
write_tsv(checks,"readiness_checks.tsv")

manifest=pd.DataFrame({"file":sorted(str(p.relative_to(PROJECT_DIR)) for p in OUT_DIR.glob("*"))})
write_tsv(manifest,"output_manifest.tsv")
run_info={"python":platform.python_version(),"conditions":CONDITION_ORDER,"alpha":ALPHA,
          "patient_condition_definition":"dataset || patient || condition",
          "note":"Diagnostic sensitivity analysis; canonical outputs are not overwritten."}
(OUT_DIR/"run_info.json").write_text(json.dumps(run_info,indent=2),encoding="utf-8")
display(checks)
print("Completed. Send the entire output directory for audit:",OUT_DIR)


,check,pass
0,all canonical inputs found,True
1,sample mapping unique,True
2,four conditions present,True
3,repeated patients detected,True
4,patient-condition compartment results written,True
5,patient-condition state results written,True
6,patient-condition program results written,True
7,patient-condition pseudobulk results written,True


Completed. Send the entire output directory for audit: G:\Repository\glioma-cnv-single-cell-atlas\metadata\sensitivity\patient_condition_sensitivity
